# 00 — Ambiente en tu propia máquina

**Todo corre aquí: el modelo, el backend y la página.** Es la versión para
trabajar solo, sin salón y sin instructor.

Hay otras dos versiones de este notebook, y conviene no mezclarlas:

| notebook | cuándo |
|---|---|
| **`00_localhost.ipynb`** | **este**: trabajas solo, todo en tu máquina |
| `00_aula.ipynb` | el taller, con el modelo en la laptop del instructor |
| `00_tec.ipynb` | el taller, con el modelo en el servidor del Tec |

Córrelo de arriba abajo. **La primera vez tarda unos minutos**, porque
construye las imágenes de Docker; las siguientes, segundos.

Las tres últimas celdas son para apagar al terminar, no para ahora.

**Requisitos:** Docker Desktop corriendo, el ambiente virtual activado y
`qwen3.5:9b` en el volumen de modelos.

## El ayudante

La celda siguiente define una función para correr comandos mostrando su salida
a medida que aparece. Sin eso, una construcción de tres minutos se ve como un
`[*]` y no hay forma de saber si avanza.

No imprime más que la ruta del repo. Córrela y sigue.

In [ ]:
import subprocess
import sys
from pathlib import Path


def raiz_del_repo():
    """
    La raiz del repo, sin importar desde donde se lanzo Jupyter.

    Puede ser notebooks/ o la raiz, segun como se abrio. Buscamos hacia arriba
    el directorio que tenga infra/ y .env.example.
    """
    p = Path.cwd().resolve()
    for candidato in (p, *p.parents):
        if (candidato / "infra").is_dir() and (candidato / ".env.example").is_file():
            return candidato
    raise RuntimeError(f"No encontre la raiz del repo desde {p}")


RAIZ = raiz_del_repo()
print("repo:", RAIZ)


def correr(comando, cwd=None, titulo=None):
    """Corre un comando e imprime su salida en vivo. Devuelve el codigo de salida."""
    if titulo:
        print(f"--- {titulo} ---")
    proceso = subprocess.Popen(
        comando,
        cwd=str(cwd or RAIZ),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # docker manda el progreso a stderr
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    for linea in proceso.stdout:
        print(linea.rstrip())
    codigo = proceso.wait()
    print(f"[codigo de salida: {codigo}]")
    return codigo

## 1. La configuración

La celda siguiente revisa el `.env` de la raíz. No lo edita: compara lo que hay
con lo que este notebook necesita —`OLLAMA_DESTINO=localhost`— y, si no
coincide, dice exactamente qué línea cambiar.

In [ ]:
# El .env de la raiz, que es UN solo archivo para todo: lo leen el codigo de
# src/paso*, estos notebooks y el contenedor del backend.
#
# Esta celda no edita nada: compara lo que hay contra lo que este notebook
# necesita, y si no coincide dice exactamente que cambiar.
import shutil

DESTINO_ESPERADO = "localhost"
MODELO_ESPERADO = "ollama:qwen3.5:9b"

env = RAIZ / ".env"
if not env.exists():
    shutil.copy(RAIZ / ".env.example", env)
    print(f"Cree {env} desde el ejemplo.")

valores = {}
for linea in env.read_text(encoding="utf-8").splitlines():
    linea = linea.strip()
    if linea and not linea.startswith("#") and "=" in linea:
        k, v = linea.split("=", 1)
        valores[k.strip()] = v.strip()

destino = valores.get("OLLAMA_DESTINO", "")
modelo = valores.get("CHAT_MODEL", "")
url = valores.get(f"OLLAMA_URL_{destino.upper()}", "(sin declarar)")

print(f"  OLLAMA_DESTINO = {destino or '(vacio)'}   ->  {url}")
print(f"  CHAT_MODEL     = {modelo or '(vacio)'}")

faltan = []
if destino != DESTINO_ESPERADO:
    faltan.append(f"OLLAMA_DESTINO={DESTINO_ESPERADO}")
if modelo != MODELO_ESPERADO:
    faltan.append(f"CHAT_MODEL={MODELO_ESPERADO}")

if faltan:
    print(f"\n  ESTE NOTEBOOK NECESITA otra configuracion. Edita el .env de la raiz:")
    for f in faltan:
        print(f"      {f}")
    print(f"  y vuelve a correr esta celda.")
else:
    print(f"\n  Configuracion correcta para este notebook.")

### Y el inventario, desde la semilla

`datos/inventario.csv` es el **estado vivo** del dominio: el backend lo reescribe
en cada `PUT`, así que no viaja en git. En un clon nuevo no existe, y si quedó
sucio de una corrida anterior, los notebooks del 02 al 06 arrancan con números
que no son los que su texto describe.

La celda siguiente lo rehace desde
[`datos/inventario.semilla.csv`](../datos/inventario.semilla.csv) **pisando lo
que hubiera**, e imprime con qué quedó. Es Python puro: no necesita Docker ni el
backend arriba, y en eso se diferencia de
`scripts/<tu-sistema>/reiniciar-datos`, que lo pide por HTTP.

> La carpeta `datos/` está montada dentro del contenedor del backend, así que
> este archivo y el que lee el backend son el mismo.

In [ ]:
# El estado vivo, rehecho desde la semilla. En un clon nuevo no existe, y si
# quedo de una corrida anterior trae los numeros de esa corrida.
import csv
import shutil

SEMILLA = RAIZ / "datos" / "inventario.semilla.csv"
VIVO = RAIZ / "datos" / "inventario.csv"

antes = f"{VIVO.stat().st_size} bytes" if VIVO.exists() else "no existia"
shutil.copy(SEMILLA, VIVO)
print(f"  copiado: {SEMILLA.name}  ->  {VIVO.name}   (antes: {antes})")
print()

with open(VIVO, encoding="utf-8", newline="") as f:
    filas = list(csv.DictReader(f))

print(f"  {'SKU':<15} {'producto':<30} {'exist.':>7} {'precio':>7}")
print(f"  {'-' * 15} {'-' * 30} {'-' * 7} {'-' * 7}")
for r in filas:
    unidades = int(r["existencias"])
    aviso = "  <- en cero" if unidades == 0 else ""
    print(f"  {r['sku']:<15} {r['nombre']:<30} {unidades:>7} "
          f"{float(r['precio']):>7.0f}{aviso}")

print(f"\n  {len(filas)} productos. Con esto arrancan los notebooks 02 al 06.")

## 2. Levantar Ollama

Las dos celdas siguientes levantan el modelo en tu GPU y esperan a que la API
conteste. Vas a ver el arranque del contenedor y después la lista de modelos.

**La primera carga de un modelo de 9 000 millones de parámetros desde disco no
es rápida:** la segunda celda espera hasta tres minutos.

No hace falta abrir el firewall: nadie más se va a conectar a tu máquina.

In [ ]:
correr(["docker", "compose", "up", "-d"],
       cwd=RAIZ / "infra" / "aula", titulo="levantando ollama")

In [ ]:
# Esperar a que la API responda, y listar lo que tiene cargado.
import time

import requests

listo = False
for intento in range(36):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        listo = True
        break
    except Exception:
        time.sleep(5)
        print(".", end="", flush=True)
print()

if not listo:
    print("La API no respondio. Revisa:  docker logs agenticai_ollama --tail 50")
else:
    for m in r.json().get("models", []):
        print(f"  {m['name']:24} {m['size'] / 2**30:.1f} GB")

## 3. Levantar el backend y la página

Son imágenes chicas, sin GPU y sin modelo. **La primera vez tarda unos
minutos** y vas a ver la construcción línea por línea.

In [ ]:
correr(["docker", "compose", "up", "-d", "--build"],
       cwd=RAIZ / "infra" / "taller", titulo="levantando backend y frontend")

## 4. Verificar

El diagnóstico completo: Python, `requests`, el `.env`, que Ollama responda y
tenga el modelo, y que el backend y la página estén arriba.

Para cada falla imprime qué hacer. Tarda unos segundos.

In [ ]:
correr([sys.executable, str(RAIZ / "scripts" / "verificar_ambiente.py")])

## 5. La primera llamada al modelo

Si devuelve texto, el camino completo funciona. **La primera llamada puede
tardar unos ocho segundos** porque el modelo se está cargando en la GPU; las
siguientes bajan a uno o dos.

In [ ]:
sys.path.insert(0, str(RAIZ / "src"))

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")
from agentlib import chat, ultima_medicion   # noqa: E402

respuesta = chat([{"role": "user", "content": "Responde solo con la palabra LISTO, sin nada mas."}])
print(respuesta["content"])
print()
print("  tokens enviados:", ultima_medicion()["tokens_enviados"])
print("  segundos       :", ultima_medicion()["segundos"])

## 6. El backend

Imprime su estado de salud y los modelos que tiene configurados. Los que
aparezcan sin `listo` están declarados en el `.env` pero les falta la clave de
API, que es información útil y no un error.

In [ ]:
import requests

print("salud  :", requests.get("http://localhost:8000/api/health", timeout=5).json())
print("modelos:")
for m in requests.get("http://localhost:8000/api/modelos", timeout=15).json()["modelos"]:
    print(f"  {'listo' if m['listo'] else '  -  '}  {m['id']}")

Y la página: **http://localhost:4200**

Déjala abierta en otra ventana. Desde el paso 3 en adelante es donde se ve el
efecto de lo que hace el agente.

---

# Al terminar: apagar

Las tres celdas siguientes son para cuando termines de trabajar. Bajar los
contenedores no borra nada: las imágenes, el volumen de modelos y los datos del
dominio sobreviven.

**Nunca uses `down -v` en `infra/aula`.** El volumen de modelos es externo y lo
comparte el proyecto itzae: borraría los 10 GB de los dos.

In [ ]:
correr(["docker", "compose", "down"],
       cwd=RAIZ / "infra" / "taller", titulo="bajando backend y frontend")

In [ ]:
correr(["docker", "compose", "down"],
       cwd=RAIZ / "infra" / "aula", titulo="bajando ollama")

In [ ]:
# Que no quedo nada arriba.
correr(["docker", "ps", "--filter", "name=agenticai",
        "--format", "{{.Names}}  {{.Status}}"], titulo="contenedores que siguen vivos")
print("(si no lista ninguno, esta todo abajo)")